### Load Prediction

In [93]:
import json
import pandas as pd
from pathlib import Path
import torch
import numpy as np
from dataset import LABELS

# TODO: Change accordingly 
testModelPred = 'domain_boundary_predictions_correct.json'
testGroundTruth = 'subset_protein_mapped_enhanced_limited_len_600.csv'

current_path = Path.cwd()
parent_2_levels_up = current_path.parents[1]
# Load predictions
pathToPred = parent_2_levels_up / 'data' / testModelPred
pathToTruth = parent_2_levels_up / 'data' / testGroundTruth
with open(pathToPred, 'r') as f:
    predictions = [json.loads(line) for line in f]
# Load ground truth
test_data = pd.read_csv(pathToTruth)
#test_data = test_data.set_index("domain_id")

In [94]:
def build_ground_truth_labels_from_df(df):
    protein_to_domains = {}

    for _, row in df.iterrows():
        protein_id = row["domain_id"]
        start = int(row["domain_start"]) - 1  # 0-based indexing
        end = int(row["domain_end"])
        if protein_id not in protein_to_domains:
            protein_to_domains[protein_id] = {
                "length": int(row["protein_length"]),
                "domains": []
            }
        protein_to_domains[protein_id]["domains"].append((start, end))

    gt_label_map = {}

    for protein_id, info in protein_to_domains.items():
        L = info["length"]
        labels = np.full(L, LABELS["NO_DOMAIN_REGION"], dtype=np.int64)

        for start, end in info["domains"]:
            length = end - start
            if length == 1:
                labels[start] = LABELS["DOMAIN_START"]
            elif length == 2:
                labels[start] = LABELS["DOMAIN_START"]
                labels[start + 1] = LABELS["DOMAIN_END"]
            else:
                labels[start] = LABELS["DOMAIN_START"]
                labels[start + 1:end - 1] = LABELS["DOMAIN_MIDDLE"]
                labels[end - 1] = LABELS["DOMAIN_END"]

        gt_label_map[protein_id] = labels

    return gt_label_map


In [95]:
ground_truth = build_ground_truth_labels_from_df(test_data)
ground_truth

{'1oksA00': array([0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 0]),
 '4dbgB02': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0]),
 '4un2B00': array([0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3,
        0, 0, 0, 0]),
 '1oaiA00': array([1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [96]:
test_data[test_data["domain_id"] == '8iahY01']

,domain_id,class,architecture,topology,homology,protein_sequence,domain_start,domain_end,cath,protein_length,protein_id
8497,8iahY01,3,80,10,10,MSYRRELEKYRDLDEDEILGALTEEELRTLENELDELDPDNALLPA...,183,347,3.80.10.10,359,8iah


### Visualize Domain Prediction

In [97]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

LABELS = {
    'NO_DOMAIN_REGION': 0,
    'DOMAIN_START': 1,
    'DOMAIN_MIDDLE': 2,
    'DOMAIN_END': 3,
}

# Setup: blue for any domain, white for no domain
cmap_colors = ['white', 'blue', 'blue', 'blue']  # Indexed by label value
custom_cmap = ListedColormap(cmap_colors)
vmin_plot = 0
vmax_plot = 3  # max label


output_dir = "output_plots"
os.makedirs(output_dir, exist_ok=True)

visualized_count = 0

for entry in predictions:
    domain_id = entry['domain_id']
    pred_arr = np.array(entry['prediction'])

    if domain_id not in test_data.index:
        continue

    row = test_data.loc[domain_id]
    protein_length = int(row['protein_length'])
    domain_start = int(row['domain_start']) - 1  # switch to 0-based
    domain_end = int(row['domain_end'])         # already inclusive

    # Ground truth label vector
    true_labels = np.full(protein_length, LABELS['NO_DOMAIN_REGION'])
    if domain_end > domain_start:
        true_labels[domain_start] = LABELS['DOMAIN_START']
        true_labels[domain_start + 1:domain_end - 1] = LABELS['DOMAIN_MIDDLE']
        true_labels[domain_end - 1] = LABELS['DOMAIN_END']

    # Resize prediction to match protein_length (if needed)
    if len(pred_arr) < protein_length:
        padded_pred = np.full(protein_length, LABELS['NO_DOMAIN_REGION'])
        padded_pred[:len(pred_arr)] = pred_arr
        pred_arr = padded_pred
    elif len(pred_arr) > protein_length:
        pred_arr = pred_arr[:protein_length]

    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
    fig.suptitle(f"Domain {domain_id} Residue-wise Prediction (Full Sequence)")

    ax1.imshow(true_labels.reshape(1, -1), cmap=custom_cmap, aspect='auto',
               extent=[0, protein_length, 0, 1], vmin=vmin_plot, vmax=vmax_plot)
    ax1.set_yticks([])
    ax1.set_title("True CATH Domains")
    ax1.set_ylabel("True")
    ax1.set_xlim(0, protein_length)

    ax2.imshow(pred_arr.reshape(1, -1), cmap=custom_cmap, aspect='auto',
               extent=[0, protein_length, 0, 1], vmin=vmin_plot, vmax=vmax_plot)
    ax2.set_yticks([])
    ax2.set_title("Predicted CATH Domains")
    ax2.set_xlabel("Residue Index")
    ax2.set_ylabel("Predicted")
    ax2.set_xlim(0, protein_length)

    # Add legend
    handles = [plt.Rectangle((0, 0), 1, 1, color=color)
               for color in ['white', 'blue']]
    ax2.legend(handles, ['No Domain', 'Domain'], loc='upper center', bbox_to_anchor=(0.5, -0.2),
               fancybox=True, shadow=True, ncol=2)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(output_dir, f"{domain_id}.png"))
    plt.close(fig)

    visualized_count += 1


### Calculate accuracy scores

In [104]:
from sklearn.metrics import accuracy_score, f1_score


def find_segments(arr):
    """Returns list of (start, end) tuples for segments > 0"""
    segments = []
    in_seg = False
    for i, val in enumerate(arr):
        if val > 0 and not in_seg:
            start = i
            in_seg = True
        elif val == 0 and in_seg:
            segments.append((start, i))
            in_seg = False
    if in_seg:
        segments.append((start, len(arr)))
    return segments

def compute_iou(gt_array, pred_array):
    """Computes Intersection over Union for domain vs no-domain"""
    gt_mask = gt_array > 0
    pred_mask = pred_array > 0

    intersection = np.logical_and(gt_mask, pred_mask).sum()
    union = np.logical_or(gt_mask, pred_mask).sum()

    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union

def compute_sov(gt_array, pred_array):
    """Computes Segment Overlap Score (SOV)"""
    gt_segments = find_segments(gt_array)
    pred_segments = find_segments(pred_array)

    total_len = sum(end - start for start, end in gt_segments)
    matched_score = 0

    for gt_start, gt_end in gt_segments:
        gt_len = gt_end - gt_start
        best_score = 0

        for pred_start, pred_end in pred_segments:
            intersect_start = max(gt_start, pred_start)
            intersect_end = min(gt_end, pred_end)
            minov = max(0, intersect_end - intersect_start)

            if minov > 0:
                maxov = max(gt_end, pred_end) - min(gt_start, pred_start)
                delta = min(
                    maxov - minov,
                    minov,
                    gt_len // 2,
                    (pred_end - pred_start) // 2
                )
                score = ((minov + delta) / maxov) * gt_len
                best_score = max(best_score, score)

        matched_score += best_score

    return matched_score / total_len if total_len > 0 else 1.0

def evaluate_predictions(predictions, ground_truth_dict):
    results = []

    for entry in predictions:
        domain_id = entry['domain_id']
        pred_arr = np.array(entry['prediction'])

        # Get ground truth array (skip if not found)
        gt_arr = ground_truth_dict[domain_id]
        if gt_arr is None:
            continue

        # Optional: truncate/pad to equal length
        min_len = min(len(gt_arr), len(pred_arr))
        gt_arr = gt_arr[:min_len]
        pred_arr = pred_arr[:min_len]

        iou = compute_iou(gt_arr, pred_arr)
        sov = compute_sov(gt_arr, pred_arr)
        accuracy = accuracy_score(gt_arr, pred_arr)
        f1 = f1_score(gt_arr, pred_arr, average='weighted') 

        results.append({
            'domain_id': domain_id,
            'prediction': pred_arr,
            'ground_truth': gt_arr,
            'iou': iou,
            'sov': sov,
            'accuracy': accuracy,
            'f1': f1
        })

    return pd.DataFrame(results)


In [105]:
complete_df = evaluate_predictions(predictions, ground_truth)
complete_df.head(3)

,domain_id,prediction,ground_truth,iou,sov,accuracy,f1
0,3ddjA02,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.287500,0.258741,0.614865,0.579060
1,2bghA01,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.316176,0.246377,0.334917,0.276333
2,1ltlA01,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 2, ...",0.023077,0.033333,0.544803,0.489437


In [106]:
# Compute average IoU and SOV over all domains
average_iou = complete_df['iou'].mean()
average_sov = complete_df['sov'].mean()
average_accuracy = complete_df['accuracy'].mean()
average_f1 = complete_df['f1'].mean()

print(f"Average IoU over all domains: {average_iou:.4f}")
print(f"Average SOV over all domains: {average_sov:.4f}")
print(f"Average accuracy over all domains: {average_accuracy:.4f}")
print(f"Average f1 over all domains: {average_f1:.4f}")

Average IoU over all domains: 0.6511
Average SOV over all domains: 0.6741
Average accuracy over all domains: 0.7905
Average f1 over all domains: 0.7836


In [108]:
def bootstrap_metric_from_df(df, metric_col='iou', n_bootstrap=1000, alpha=0.05):
    scores = df[metric_col].values
    n = len(scores)
    boot_means = []

    for _ in range(n_bootstrap):
        sample_indices = np.random.choice(n, size=n, replace=True)
        sample_scores = scores[sample_indices]
        boot_means.append(np.mean(sample_scores))

    lower = np.percentile(boot_means, 100 * (alpha / 2))
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))
    mean = np.mean(scores)

    return mean, lower, upper

# Example usage with your DataFrame `df`:
mean_iou, ci_lower_iou, ci_upper_iou = bootstrap_metric_from_df(complete_df, metric_col='iou')
mean_sov, ci_lower_sov, ci_upper_sov = bootstrap_metric_from_df(complete_df, metric_col='sov')
mean_acc, ci_lower_acc, ci_upper_acc = bootstrap_metric_from_df(complete_df, metric_col='accuracy')
mean_f1, ci_lower_f1, ci_upper_f1 = bootstrap_metric_from_df(complete_df, metric_col='f1')

print(f"IoU: mean={mean_iou:.4f}, 95% CI=({ci_lower_iou:.4f}, {ci_upper_iou:.4f})")
print(f"SOV: mean={mean_sov:.4f}, 95% CI=({ci_lower_sov:.4f}, {ci_upper_sov:.4f})")
print(f"Accuracy: mean={mean_acc:.4f}, 95% CI=({ci_lower_acc:.4f}, {ci_upper_acc:.4f})")
print(f"F1: mean={mean_f1:.4f}, 95% CI=({ci_lower_f1:.4f}, {ci_upper_f1:.4f})")


IoU: mean=0.6511, 95% CI=(0.6278, 0.6750)
SOV: mean=0.6741, 95% CI=(0.6510, 0.6987)
Accuracy: mean=0.7905, 95% CI=(0.7775, 0.8052)
F1: mean=0.7836, 95% CI=(0.7708, 0.7968)
